# Reproducible tuning and model persistence

Choose Ridge regularization with training-only cross-validation, evaluate a held-out test set, and save the complete fitted pipeline. Reload the model and verify that predictions are unchanged.

The generated dataset is a teaching example. A temporary directory keeps this lesson from overwriting project artifacts.


In [ ]:
import json
import tempfile
from pathlib import Path
import joblib
import numpy as np
import sklearn
from sklearn.datasets import make_regression
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.model_selection import GridSearchCV, KFold, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

SEED = 42
X, y = make_regression(n_samples=320, n_features=8, n_informative=5,
    noise=15.0, random_state=SEED)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=SEED)
model = Pipeline([('scale', StandardScaler()), ('regress', Ridge())])


## 1. Tune on the training split

Negative mean squared error is maximized by scikit-learn's search. Negate it to recover MSE. Taking its square root here reports root mean of fold MSEs, not a separate mean of fold RMSEs.


In [ ]:
search = GridSearchCV(model, {'regress__alpha': [0.01, 0.1, 1.0, 10.0, 100.0]},
    cv=KFold(n_splits=4, shuffle=True, random_state=SEED),
    scoring='neg_mean_squared_error', n_jobs=1, refit=True)
search.fit(X_train, y_train)
print('Best settings:', search.best_params_)
print('Root mean fold MSE:', round(np.sqrt(-search.best_score_), 3))
prediction = search.predict(X_test)
print('Held-out RMSE:', round(np.sqrt(mean_squared_error(y_test, prediction)), 3))
print('Held-out R²:', round(r2_score(y_test, prediction), 3))


## 2. Save and reload the fitted pipeline

Only the local artifact created in this cell is loaded. The pipeline includes the training scaler, which keeps prediction preprocessing consistent. Metadata records the seed, feature count, selected settings, and package version; it does not replace a data/versioning strategy for a real deployment.


In [ ]:
metadata = {'seed': SEED, 'features': X.shape[1],
            'scikit_learn': sklearn.__version__, 'settings': search.best_params_}
with tempfile.TemporaryDirectory(prefix='ridge_lesson_') as folder:
    folder = Path(folder)
    model_path = folder / 'ridge_pipeline.joblib'
    joblib.dump(search.best_estimator_, model_path)
    (folder / 'metadata.json').write_text(json.dumps(metadata, indent=2))
    restored = joblib.load(model_path)
    np.testing.assert_allclose(restored.predict(X_test), prediction, rtol=1e-12, atol=1e-12)
    print('Reloaded predictions match; saved bytes:', model_path.stat().st_size)
    print((folder / 'metadata.json').read_text())


## Practice

Add a candidate model while keeping the same training folds. For an application, store the artifact under `artifacts/` with a versioned filename and document its required feature order. Use the [Adult Income project](../../../projects/adult_income/README.md) for a fuller training/serving workflow.

**References:** [parameter search](https://scikit-learn.org/stable/modules/grid_search.html), [model persistence](https://scikit-learn.org/stable/model_persistence.html).
